# FMA Genre Classifier (Transfer Learning with Pretrained MERT)

This notebook keeps the same data/split policy style as `genre_classifier_workflow.ipynb` but replaces the CNN with a pretrained audio encoder (`MERT`) + classification head.

Notes:
- This is still **single-label top-genre** training (`track.genre_top`), like your current notebook.
- It supports `official` and `stratified` splits, rare-class merge policy, macro-metric early stopping, and multi-crop eval.


In [6]:
from __future__ import annotations

import json
import random
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from transformers import AutoFeatureExtractor, AutoModel
except Exception as exc:
    raise RuntimeError(
        "Missing transformers dependency. Install with: pip install transformers accelerate"
    ) from exc


In [7]:
# ===== Config =====
AUDIO_DIR = Path('fma_large')
METADATA_PATH = Path('fma_metadata/tracks.csv')
OUTPUT_DIR = Path('outputs/transfer_mert_run')

# Pretrained encoder
PRETRAINED_MODEL_NAME = 'm-a-p/MERT-v1-95M'
TRUST_REMOTE_CODE = True  # MERT repo uses custom model code on Hugging Face
FREEZE_ENCODER = True
UNFREEZE_LAST_N_LAYERS = 0  # used only when FREEZE_ENCODER=False
HEAD_DROPOUT = 0.3

# Audio/crop settings
AUDIO_MAX_DURATION = 29.0   # cap decoded duration for each file
TRAIN_CROP_DURATION = 5.0
EVAL_CROP_DURATION = 5.0
VAL_MULTI_CROPS = 5
TEST_MULTI_CROPS = 7

# Train settings
EPOCHS = 18
BATCH_SIZE = 12
LR_HEAD = 1e-3
LR_ENCODER = 1e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 42
DEVICE_PREF = 'auto'

MAX_TRAIN = None
MAX_VAL = None
MAX_TEST = None

SPLIT_STRATEGY = 'stratified'  # 'stratified' or 'official'
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
MIN_CLASS_COUNT_FOR_SPLIT = 10
MIN_TRAIN_COUNT_FOR_MODEL = 50
MERGE_RARE_CLASSES = True
RARE_CLASS_NAME = 'Other'

MIN_AUDIO_FILE_BYTES = 4096
EXCLUDE_BAD_AUDIO_SCAN = True
BAD_AUDIO_SCAN_PATH = Path('outputs/bad_audio_scan.csv')

USE_BALANCED_SAMPLER = True
USE_CLASS_WEIGHTED_LOSS = False  # leave False when sampler=True to avoid double balancing

EARLY_STOP_PATIENCE = 6
METRIC_FOR_BEST = 'val_f1_macro'  # or 'val_recall_macro'

# Logging/debug settings
TRAIN_LOG_EVERY = 20
EVAL_LOG_EVERY = 200
DATALOADER_SANITY_BATCHES = 2
MAX_DECODE_RETRIES = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)


OUTPUT_DIR: outputs/transfer_mert_run


In [8]:
# ===== Utilities =====
SPLIT_TRAIN = 'training'
SPLIT_VAL = 'validation'
SPLIT_TEST = 'test'

@dataclass
class TrackRecord:
    track_id: int
    genre: str
    split: str


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def choose_device(device_arg: str) -> torch.device:
    if device_arg == 'cpu':
        return torch.device('cpu')
    if device_arg == 'cuda':
        if not torch.cuda.is_available():
            raise ValueError('Requested cuda but CUDA unavailable.')
        return torch.device('cuda')
    if device_arg == 'mps':
        if not torch.backends.mps.is_available():
            raise ValueError('Requested mps but MPS unavailable.')
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def optimize_runtime(device: torch.device) -> Dict[str, object]:
    use_amp = False
    amp_dtype = None
    torch.set_float32_matmul_precision('high')

    if device.type == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        use_amp = True
        amp_dtype = torch.float16

    return {'use_amp': use_amp, 'amp_dtype': amp_dtype}


def track_path(audio_dir: Path, track_id: int) -> Path:
    return audio_dir / f"{track_id:06d}"[:3] / f"{track_id:06d}.mp3"


def load_records(metadata_path: Path, audio_dir: Path, min_audio_file_bytes: int = 0) -> List[TrackRecord]:
    df = pd.read_csv(metadata_path, header=[0, 1], index_col=0, low_memory=False)
    df.index = df.index.astype(int)

    subset_mask = df[('set', 'subset')] == 'large'
    labeled_mask = df[('track', 'genre_top')].notna()
    filtered = df[subset_mask & labeled_mask]

    out: List[TrackRecord] = []
    skipped_missing = 0
    skipped_tiny = 0

    for track_id, row in filtered.iterrows():
        ap = track_path(audio_dir, int(track_id))
        if not ap.exists():
            skipped_missing += 1
            continue
        if min_audio_file_bytes > 0 and ap.stat().st_size < min_audio_file_bytes:
            skipped_tiny += 1
            continue
        out.append(TrackRecord(int(track_id), str(row[('track', 'genre_top')]), str(row[('set', 'split')])))

    print(f'Loaded labeled records={len(out)} (skipped missing={skipped_missing}, tiny<{min_audio_file_bytes}B={skipped_tiny})')
    return out


def drop_known_bad_audio(records: Sequence[TrackRecord], bad_audio_scan_path: Path, enabled: bool) -> List[TrackRecord]:
    if not enabled:
        return list(records)
    if not bad_audio_scan_path.exists():
        print(f'Bad-audio scan not found: {bad_audio_scan_path}; continuing without extra filtering.')
        return list(records)

    bad = pd.read_csv(bad_audio_scan_path)
    if 'track_id' not in bad.columns:
        print(f'Bad-audio scan missing track_id column: {bad_audio_scan_path}; continuing without extra filtering.')
        return list(records)

    bad_ids = set(bad['track_id'].astype(int).tolist())
    filtered = [r for r in records if r.track_id not in bad_ids]
    print(f'Dropped known bad-audio tracks from scan: {len(records) - len(filtered)}')
    return filtered


def split_records_official(records: Sequence[TrackRecord]) -> Tuple[List[TrackRecord], List[TrackRecord], List[TrackRecord]]:
    train = [TrackRecord(r.track_id, r.genre, SPLIT_TRAIN) for r in records if r.split == SPLIT_TRAIN]
    val = [TrackRecord(r.track_id, r.genre, SPLIT_VAL) for r in records if r.split == SPLIT_VAL]
    test = [TrackRecord(r.track_id, r.genre, SPLIT_TEST) for r in records if r.split == SPLIT_TEST]
    return train, val, test


def split_records_stratified(
    records: Sequence[TrackRecord],
    seed: int,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    min_class_count: int,
) -> Tuple[List[TrackRecord], List[TrackRecord], List[TrackRecord], Dict[str, int]]:
    total = train_ratio + val_ratio + test_ratio
    if abs(total - 1.0) > 1e-8:
        raise ValueError(f'Train/val/test ratios must sum to 1.0, got {total}')

    df = pd.DataFrame([{'track_id': r.track_id, 'genre': r.genre, 'split': r.split} for r in records])
    counts = df['genre'].value_counts()

    keep_genres = counts[counts >= min_class_count].index.tolist()
    dropped = counts[counts < min_class_count].to_dict()
    df = df[df['genre'].isin(keep_genres)].copy()

    if len(df) == 0:
        raise RuntimeError('No records left after min_class_count filtering; lower MIN_CLASS_COUNT_FOR_SPLIT.')

    idx = np.arange(len(df))
    y = df['genre'].to_numpy()

    train_idx, hold_idx = train_test_split(
        idx,
        test_size=(1.0 - train_ratio),
        random_state=seed,
        shuffle=True,
        stratify=y,
    )

    hold_y = y[hold_idx]
    hold_test_ratio = test_ratio / (val_ratio + test_ratio)
    val_idx, test_idx = train_test_split(
        hold_idx,
        test_size=hold_test_ratio,
        random_state=seed + 1,
        shuffle=True,
        stratify=hold_y,
    )

    def build(ix: np.ndarray, split_name: str) -> List[TrackRecord]:
        chunk = df.iloc[ix]
        return [TrackRecord(int(r.track_id), str(r.genre), split_name) for r in chunk.itertuples(index=False)]

    train = build(train_idx, SPLIT_TRAIN)
    val = build(val_idx, SPLIT_VAL)
    test = build(test_idx, SPLIT_TEST)
    return train, val, test, dropped


def maybe_limit(records: List[TrackRecord], max_items: int | None, seed: int) -> List[TrackRecord]:
    if max_items is None or len(records) <= max_items:
        return records
    rng = random.Random(seed)
    x = records.copy()
    rng.shuffle(x)
    return x[:max_items]


def apply_train_support_policy(
    train_records: Sequence[TrackRecord],
    val_records: Sequence[TrackRecord],
    test_records: Sequence[TrackRecord],
    min_train_count: int,
    merge_rare: bool,
    rare_class_name: str,
) -> Tuple[List[TrackRecord], List[TrackRecord], List[TrackRecord], Dict[str, object]]:
    train = list(train_records)
    val = list(val_records)
    test = list(test_records)

    def counts_of(rs: Sequence[TrackRecord]) -> pd.Series:
        if len(rs) == 0:
            return pd.Series(dtype='int64')
        return pd.Series([r.genre for r in rs]).value_counts().sort_index()

    def relabel(rs: Sequence[TrackRecord], remap: Dict[str, str]) -> List[TrackRecord]:
        out: List[TrackRecord] = []
        for r in rs:
            g = remap.get(r.genre, r.genre)
            out.append(TrackRecord(r.track_id, g, r.split))
        return out

    info: Dict[str, object] = {
        'min_train_count': int(min_train_count),
        'initial_train_counts': counts_of(train).astype(int).to_dict(),
        'rare_classes': {},
        'merged_to': None,
        'dropped_after_policy': {},
        'final_train_counts': {},
    }

    train_counts = counts_of(train)
    rare = train_counts[train_counts < min_train_count]
    info['rare_classes'] = rare.astype(int).to_dict()

    if merge_rare and len(rare) > 0:
        remap = {g: rare_class_name for g in rare.index.tolist()}
        train = relabel(train, remap)
        val = relabel(val, remap)
        test = relabel(test, remap)
        info['merged_to'] = rare_class_name

    post_counts = counts_of(train)
    keep = set(post_counts[post_counts >= min_train_count].index.tolist())
    dropped = post_counts[~post_counts.index.isin(keep)]
    info['dropped_after_policy'] = dropped.astype(int).to_dict()

    if len(dropped) > 0:
        train = [r for r in train if r.genre in keep]
        val = [r for r in val if r.genre in keep]
        test = [r for r in test if r.genre in keep]

    final_counts = counts_of(train)
    if len(final_counts) == 0:
        raise RuntimeError('No classes left after train-support policy; lower MIN_TRAIN_COUNT_FOR_MODEL.')
    info['final_train_counts'] = final_counts.astype(int).to_dict()
    return train, val, test, info


def print_split_support(train_records, val_records, test_records, min_train_count):
    for split_name, rs in [('train', train_records), ('val', val_records), ('test', test_records)]:
        counts = pd.Series([r.genre for r in rs]).value_counts().sort_index()
        print(f'\n[{split_name}] classes={len(counts)} min={int(counts.min())} median={float(counts.median()):.1f} max={int(counts.max())}')
        print(counts.to_string())

    all_counts = pd.Series([r.genre for r in train_records]).value_counts().sort_index()
    tiny = all_counts[all_counts < min_train_count]
    if len(tiny) > 0:
        print(f'\nWarning: very low-support classes in train (<{min_train_count} samples).')
        print(tiny.to_string())


class WaveCropDataset(Dataset):
    def __init__(
        self,
        records: Sequence[TrackRecord],
        audio_dir: Path,
        label_to_idx: Dict[str, int],
        sample_rate: int,
        crop_duration: float,
        max_duration: float,
        train_mode: bool,
        max_decode_retries: int = 1,
        decode_error_log_every: int = 50,
    ):
        self.records = list(records)
        self.audio_dir = audio_dir
        self.label_to_idx = label_to_idx
        self.sample_rate = sample_rate
        self.crop_duration = crop_duration
        self.max_duration = max_duration
        self.train_mode = train_mode
        self.max_decode_retries = int(max_decode_retries)
        self.decode_error_log_every = int(max(1, decode_error_log_every))
        self.decode_error_count = 0
        self.crop_samples = int(round(sample_rate * crop_duration))

    def __len__(self):
        return len(self.records)

    def _load_crop(self, rec: TrackRecord):
        ap = track_path(self.audio_dir, rec.track_id)

        y = None
        last_exc = None
        for _ in range(max(1, self.max_decode_retries + 1)):
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings('ignore', message='PySoundFile failed. Trying audioread instead.')
                    warnings.filterwarnings('ignore', message='librosa.core.audio.__audioread_load')
                    y, _ = librosa.load(
                        str(ap),
                        sr=self.sample_rate,
                        mono=True,
                        duration=float(self.max_duration),
                        res_type='kaiser_fast',
                    )
                break
            except Exception as exc:
                last_exc = exc
                y = None

        if y is None or y.shape[0] == 0:
            self.decode_error_count += 1
            if self.decode_error_count <= 5 or self.decode_error_count % self.decode_error_log_every == 0:
                print(
                    f"decode fallback #{self.decode_error_count}: track_id={rec.track_id} "
                    f"path={ap} err={type(last_exc).__name__ if last_exc else 'empty_audio'}"
                )
            y = np.zeros(self.crop_samples, dtype=np.float32)

        total = y.shape[0]
        if total <= self.crop_samples:
            if total < self.crop_samples:
                y = np.pad(y, (0, self.crop_samples - total))
            return y.astype(np.float32)

        if self.train_mode:
            start = random.randint(0, total - self.crop_samples)
        else:
            start = (total - self.crop_samples) // 2

        crop = y[start : start + self.crop_samples]
        return crop.astype(np.float32)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        wav = self._load_crop(rec)
        y = self.label_to_idx[rec.genre]
        return wav, y


def compute_metrics(y_true: Sequence[int], y_pred: Sequence[int], num_classes: int) -> Dict[str, float]:
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(num_classes)), average='macro', zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(num_classes)), average='weighted', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(p_macro),
        'recall_macro': float(r_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(p_weighted),
        'recall_weighted': float(r_weighted),
        'f1_weighted': float(f1_weighted),
    }


class MERTGenreClassifier(nn.Module):
    def __init__(self, encoder_name: str, num_classes: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, trust_remote_code=TRUST_REMOTE_CODE)
        hidden = int(self.encoder.config.hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        out = self.encoder(input_values=input_values, attention_mask=attention_mask)
        h = out.last_hidden_state
        pooled = h.mean(dim=1)
        return self.head(self.dropout(pooled))


def class_weights(records: Sequence[TrackRecord], label_to_idx: Dict[str, int], device: torch.device) -> torch.Tensor:
    counts = np.zeros(len(label_to_idx), dtype=np.float64)
    for r in records:
        counts[label_to_idx[r.genre]] += 1
    counts = np.maximum(counts, 1.0)
    inv = 1.0 / counts
    w = inv / inv.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)


def set_encoder_trainability(model: MERTGenreClassifier, freeze_encoder: bool, unfreeze_last_n_layers: int):
    for p in model.encoder.parameters():
        p.requires_grad = not freeze_encoder

    if freeze_encoder:
        return

    # Optional partial unfreeze for Wav2Vec2-style encoder stacks.
    if unfreeze_last_n_layers > 0 and hasattr(model.encoder, 'encoder') and hasattr(model.encoder.encoder, 'layers'):
        layers = list(model.encoder.encoder.layers)
        cutoff = max(0, len(layers) - unfreeze_last_n_layers)
        for i, layer in enumerate(layers):
            req = i >= cutoff
            for p in layer.parameters():
                p.requires_grad = req


def build_collate(feature_extractor, sample_rate: int):
    def collate(batch):
        wavs, ys = zip(*batch)
        enc = feature_extractor(
            list(wavs),
            sampling_rate=sample_rate,
            return_tensors='pt',
            padding=True,
        )
        labels = torch.tensor(ys, dtype=torch.long)
        return enc.input_values, enc.attention_mask, labels
    return collate


def sanity_check_dataloader(loader, max_batches: int = 2):
    print(f'Running dataloader sanity check for {max_batches} batch(es)...')
    t0 = time.perf_counter()
    for i, (input_values, attention_mask, yb) in enumerate(loader, start=1):
        dt = time.perf_counter() - t0
        print(
            f'sanity batch {i}: input_values={tuple(input_values.shape)} '
            f'attention_mask={tuple(attention_mask.shape)} labels={tuple(yb.shape)} '
            f'elapsed={dt:.2f}s'
        )
        if i >= max_batches:
            break



def run_train_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    amp_enabled,
    amp_dtype,
    train_log_every: int = 20,
    epoch: int | None = None,
):
    model.train(True)
    total_loss = 0.0
    seen = 0
    y_true, y_pred = [], []

    total_batches = len(loader)
    t0 = time.perf_counter()

    for batch_idx, (input_values, attention_mask, yb) in enumerate(loader, start=1):
        input_values = input_values.to(device)
        attention_mask = attention_mask.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        if amp_enabled:
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=True):
                logits = model(input_values, attention_mask)
                loss = criterion(logits, yb)
        else:
            logits = model(input_values, attention_mask)
            loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        batch_n = int(input_values.size(0))
        seen += batch_n
        total_loss += float(loss.item()) * batch_n

        pred = torch.argmax(logits, dim=1)
        y_true.extend(yb.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())

        if train_log_every and (batch_idx == 1 or batch_idx % train_log_every == 0 or batch_idx == total_batches):
            elapsed = max(1e-6, time.perf_counter() - t0)
            it_per_sec = batch_idx / elapsed
            eta_sec = max(0.0, (total_batches - batch_idx) / max(1e-6, it_per_sec))
            avg_loss = total_loss / max(1, seen)
            prefix = f"epoch {epoch:02d} " if epoch is not None else ""
            print(
                f"{prefix}train batch {batch_idx}/{total_batches} "
                f"loss={float(loss.item()):.4f} avg_loss={avg_loss:.4f} "
                f"it/s={it_per_sec:.2f} eta={eta_sec/60.0:.1f}m"
            )

    return total_loss / max(1, len(loader.dataset)), y_true, y_pred


def predict_multicrop_track(
    model,
    feature_extractor,
    rec: TrackRecord,
    audio_dir: Path,
    device: torch.device,
    sample_rate: int,
    max_duration: float,
    crop_duration: float,
    num_crops: int,
):
    ap = track_path(audio_dir, rec.track_id)
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='PySoundFile failed. Trying audioread instead.')
            warnings.filterwarnings('ignore', message='librosa.core.audio.__audioread_load')
            y, _ = librosa.load(
                str(ap),
                sr=sample_rate,
                mono=True,
                duration=float(max_duration),
                res_type='kaiser_fast',
            )
    except Exception:
        y = np.zeros(int(sample_rate * crop_duration), dtype=np.float32)

    if y is None or y.shape[0] == 0:
        y = np.zeros(int(sample_rate * crop_duration), dtype=np.float32)
    y = y.astype(np.float32)

    crop_samples = int(round(sample_rate * crop_duration))
    total = y.shape[0]

    if total <= crop_samples:
        starts = [0]
    else:
        starts = np.linspace(0, total - crop_samples, num=max(1, num_crops), dtype=int).tolist()

    wavs = []
    for s in starts:
        crop = y[s : s + crop_samples]
        if crop.shape[0] < crop_samples:
            crop = np.pad(crop, (0, crop_samples - crop.shape[0]))
        wavs.append(crop.astype(np.float32))

    enc = feature_extractor(wavs, sampling_rate=sample_rate, return_tensors='pt', padding=True)
    input_values = enc.input_values.to(device)
    attention_mask = enc.attention_mask.to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_values, attention_mask)
        probs = torch.softmax(logits, dim=1)
        mean_prob = probs.mean(dim=0)
    return mean_prob


def evaluate_multicrop(
    model,
    feature_extractor,
    records: Sequence[TrackRecord],
    label_to_idx: Dict[str, int],
    audio_dir: Path,
    device: torch.device,
    sample_rate: int,
    max_duration: float,
    crop_duration: float,
    num_crops: int,
    log_every: int = 500,
):
    y_true, y_pred = [], []
    for i, rec in enumerate(records, start=1):
        prob = predict_multicrop_track(
            model,
            feature_extractor,
            rec,
            audio_dir,
            device,
            sample_rate,
            max_duration,
            crop_duration,
            num_crops,
        )
        pred = int(torch.argmax(prob).item())
        y_true.append(label_to_idx[rec.genre])
        y_pred.append(pred)
        if (log_every and i % log_every == 0) or i == len(records):
            print(f'eval progress: {i}/{len(records)}')

    return compute_metrics(y_true, y_pred, len(label_to_idx)), y_true, y_pred


In [9]:
# ===== Prepare splits =====
records = load_records(METADATA_PATH, AUDIO_DIR, min_audio_file_bytes=MIN_AUDIO_FILE_BYTES)
records = drop_known_bad_audio(records, BAD_AUDIO_SCAN_PATH, enabled=EXCLUDE_BAD_AUDIO_SCAN)
if not records:
    raise RuntimeError('No labeled tracks found after filtering.')

if SPLIT_STRATEGY == 'official':
    train_records, val_records, test_records = split_records_official(records)
    dropped_classes = {}
elif SPLIT_STRATEGY == 'stratified':
    train_records, val_records, test_records, dropped_classes = split_records_stratified(
        records,
        seed=SEED,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        min_class_count=MIN_CLASS_COUNT_FOR_SPLIT,
    )
else:
    raise ValueError(f'Unknown SPLIT_STRATEGY: {SPLIT_STRATEGY}')

train_records = maybe_limit(train_records, MAX_TRAIN, SEED)
val_records = maybe_limit(val_records, MAX_VAL, SEED + 1)
test_records = maybe_limit(test_records, MAX_TEST, SEED + 2)

train_records, val_records, test_records, support_policy = apply_train_support_policy(
    train_records,
    val_records,
    test_records,
    min_train_count=MIN_TRAIN_COUNT_FOR_MODEL,
    merge_rare=MERGE_RARE_CLASSES,
    rare_class_name=RARE_CLASS_NAME,
)

print(f'split_strategy={SPLIT_STRATEGY}')
print(f'train={len(train_records)} val={len(val_records)} test={len(test_records)}')
if dropped_classes:
    print('Dropped low-count classes for stratified splitting:')
    print(pd.Series(dropped_classes).sort_values().to_string())
if support_policy.get('rare_classes'):
    print(f"Rare classes in train (<{MIN_TRAIN_COUNT_FOR_MODEL}):")
    print(pd.Series(support_policy['rare_classes']).sort_values().to_string())
if support_policy.get('merged_to') is not None:
    print(f"Merged rare classes into: {support_policy['merged_to']}")
if support_policy.get('dropped_after_policy'):
    print('Dropped classes after train-support policy:')
    print(pd.Series(support_policy['dropped_after_policy']).sort_values().to_string())

print_split_support(train_records, val_records, test_records, MIN_TRAIN_COUNT_FOR_MODEL)


KeyboardInterrupt: 

In [ ]:
# ===== Train =====
set_seed(SEED)
device = choose_device(DEVICE_PREF)
runtime = optimize_runtime(device)

feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED_MODEL_NAME, trust_remote_code=TRUST_REMOTE_CODE)
MODEL_SAMPLE_RATE = int(feature_extractor.sampling_rate)
print('Model sample rate:', MODEL_SAMPLE_RATE)

all_genres = sorted({r.genre for r in [*train_records, *val_records, *test_records]})
label_to_idx = {g: i for i, g in enumerate(all_genres)}
idx_to_label = {i: g for g, i in label_to_idx.items()}

train_ds = WaveCropDataset(
    train_records,
    AUDIO_DIR,
    label_to_idx,
    sample_rate=MODEL_SAMPLE_RATE,
    crop_duration=TRAIN_CROP_DURATION,
    max_duration=AUDIO_MAX_DURATION,
    train_mode=True,
    max_decode_retries=MAX_DECODE_RETRIES,
)

collate_fn = build_collate(feature_extractor, MODEL_SAMPLE_RATE)

if USE_BALANCED_SAMPLER:
    counts = np.zeros(len(label_to_idx), dtype=np.float64)
    for r in train_records:
        counts[label_to_idx[r.genre]] += 1
    sample_weights = [1.0 / counts[label_to_idx[r.genre]] for r in train_records]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_records), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, collate_fn=collate_fn)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)

# Quick check so data loading/decoding issues surface immediately.
sanity_check_dataloader(train_loader, max_batches=DATALOADER_SANITY_BATCHES)

model = MERTGenreClassifier(PRETRAINED_MODEL_NAME, num_classes=len(all_genres), dropout=HEAD_DROPOUT).to(device)
set_encoder_trainability(model, freeze_encoder=FREEZE_ENCODER, unfreeze_last_n_layers=UNFREEZE_LAST_N_LAYERS)

cw = class_weights(train_records, label_to_idx, device)
ce_weight = cw if USE_CLASS_WEIGHTED_LOSS and (not USE_BALANCED_SAMPLER) else None
criterion = nn.CrossEntropyLoss(weight=ce_weight)

head_params = [p for p in model.head.parameters() if p.requires_grad]
encoder_params = [p for p in model.encoder.parameters() if p.requires_grad]

param_groups = []
if head_params:
    param_groups.append({'params': head_params, 'lr': LR_HEAD})
if encoder_params:
    param_groups.append({'params': encoder_params, 'lr': LR_ENCODER})

optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

history = []
best_score = -1.0
best_ckpt = OUTPUT_DIR / 'best_model.pt'
no_improve = 0

print('Using device:', device)
print('Runtime:', runtime)
print('Classes:', len(all_genres))
print('Encoder:', PRETRAINED_MODEL_NAME)
print('FREEZE_ENCODER:', FREEZE_ENCODER)
print('UNFREEZE_LAST_N_LAYERS:', UNFREEZE_LAST_N_LAYERS)

for epoch in range(1, EPOCHS + 1):
    tr_loss, ytr, ptr = run_train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        runtime['use_amp'],
        runtime['amp_dtype'],
        train_log_every=TRAIN_LOG_EVERY,
        epoch=epoch,
    )
    tr_metrics = compute_metrics(ytr, ptr, len(all_genres))

    val_metrics, _, _ = evaluate_multicrop(
        model,
        feature_extractor,
        val_records,
        label_to_idx,
        AUDIO_DIR,
        device,
        sample_rate=MODEL_SAMPLE_RATE,
        max_duration=AUDIO_MAX_DURATION,
        crop_duration=EVAL_CROP_DURATION,
        num_crops=VAL_MULTI_CROPS,
        log_every=EVAL_LOG_EVERY,
    )

    scheduler.step(val_metrics['f1_macro'])

    row = {
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_accuracy': tr_metrics['accuracy'],
        'train_precision_macro': tr_metrics['precision_macro'],
        'train_recall_macro': tr_metrics['recall_macro'],
        'train_f1_macro': tr_metrics['f1_macro'],
        'val_accuracy': val_metrics['accuracy'],
        'val_precision_macro': val_metrics['precision_macro'],
        'val_recall_macro': val_metrics['recall_macro'],
        'val_f1_macro': val_metrics['f1_macro'],
        'val_precision_weighted': val_metrics['precision_weighted'],
        'val_recall_weighted': val_metrics['recall_weighted'],
        'val_f1_weighted': val_metrics['f1_weighted'],
        'lr_head': float(optimizer.param_groups[0]['lr']),
    }
    history.append(row)

    score = row[METRIC_FOR_BEST]
    print(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"train_f1={row['train_f1_macro']:.4f} "
        f"val_f1={row['val_f1_macro']:.4f} "
        f"val_prec={row['val_precision_macro']:.4f} "
        f"val_rec={row['val_recall_macro']:.4f}"
    )

    if score > best_score:
        best_score = score
        no_improve = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_to_idx': label_to_idx,
            'idx_to_label': idx_to_label,
            'config': {
                'pretrained_model_name': PRETRAINED_MODEL_NAME,
                'model_sample_rate': MODEL_SAMPLE_RATE,
                'audio_max_duration': AUDIO_MAX_DURATION,
                'train_crop_duration': TRAIN_CROP_DURATION,
                'eval_crop_duration': EVAL_CROP_DURATION,
                'val_multi_crops': VAL_MULTI_CROPS,
                'test_multi_crops': TEST_MULTI_CROPS,
                'freeze_encoder': FREEZE_ENCODER,
                'unfreeze_last_n_layers': UNFREEZE_LAST_N_LAYERS,
                'head_dropout': HEAD_DROPOUT,
            },
            'epoch': epoch,
            'best_metric': METRIC_FOR_BEST,
            'best_score': best_score,
            'val_metrics': val_metrics,
        }, best_ckpt)
    else:
        no_improve += 1

    if no_improve >= EARLY_STOP_PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

hist_df = pd.DataFrame(history)
hist_df.to_csv(OUTPUT_DIR / 'history.csv', index=False)
print('Training complete. Best score:', best_score)


Model sample rate: 24000


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 42877.82it/s]


Using device: mps
Runtime: {'use_amp': False, 'amp_dtype': None}
Classes: 12
Encoder: m-a-p/MERT-v1-95M
FREEZE_ENCODER: True
UNFREEZE_LAST_N_LAYERS: 0


`use_return_dict` is deprecated! Use `return_dict` instead!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1776] error: part2_3_length (2112) too large for available bit count (1984)


KeyboardInterrupt: 

In [ ]:
# ===== Final evaluation + reporting =====
ckpt = torch.load(OUTPUT_DIR / 'best_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

ckpt_label_to_idx = ckpt['label_to_idx']
ckpt_idx_to_label = ckpt.get('idx_to_label', {v: k for k, v in ckpt_label_to_idx.items()})

val_metrics, _, _ = evaluate_multicrop(
    model,
    feature_extractor,
    val_records,
    ckpt_label_to_idx,
    AUDIO_DIR,
    device,
    sample_rate=MODEL_SAMPLE_RATE,
    max_duration=AUDIO_MAX_DURATION,
    crop_duration=EVAL_CROP_DURATION,
    num_crops=VAL_MULTI_CROPS,
)

test_metrics, y_te, p_te = evaluate_multicrop(
    model,
    feature_extractor,
    test_records,
    ckpt_label_to_idx,
    AUDIO_DIR,
    device,
    sample_rate=MODEL_SAMPLE_RATE,
    max_duration=AUDIO_MAX_DURATION,
    crop_duration=EVAL_CROP_DURATION,
    num_crops=TEST_MULTI_CROPS,
)

cm = confusion_matrix(y_te, p_te, labels=list(range(len(ckpt_label_to_idx))))
np.save(OUTPUT_DIR / 'test_confusion_matrix.npy', cm)

summary = {
    'best_metric': ckpt['best_metric'],
    'best_score': ckpt['best_score'],
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'checkpoint': str(OUTPUT_DIR / 'best_model.pt'),
}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('VAL:', val_metrics)
print('TEST:', test_metrics)
print('Classification report (test):')
print(
    classification_report(
        y_te,
        p_te,
        labels=list(range(len(ckpt_label_to_idx))),
        target_names=[ckpt_idx_to_label[i] for i in range(len(ckpt_label_to_idx))],
        digits=4,
        zero_division=0,
    )
)

hist = pd.read_csv(OUTPUT_DIR / 'history.csv')
display(hist.tail())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist['epoch'], hist['train_f1_macro'], label='train_f1_macro')
ax[0].plot(hist['epoch'], hist['val_f1_macro'], label='val_f1_macro')
ax[0].set_title('Macro F1')
ax[0].legend()

ax[1].plot(hist['epoch'], hist['val_precision_macro'], label='val_precision_macro')
ax[1].plot(hist['epoch'], hist['val_recall_macro'], label='val_recall_macro')
ax[1].set_title('Macro Precision / Recall')
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ===== Single-file inference helper =====
def predict_file(audio_file: str, checkpoint_path: Path, top_k: int = 5):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    l2i = checkpoint['label_to_idx']
    i2l = checkpoint.get('idx_to_label', {v:k for k,v in l2i.items()})
    cfg = checkpoint.get('config', {})

    model_name = str(cfg.get('pretrained_model_name', PRETRAINED_MODEL_NAME))
    sr = int(cfg.get('model_sample_rate', MODEL_SAMPLE_RATE))
    max_dur = float(cfg.get('audio_max_duration', AUDIO_MAX_DURATION))
    crop_dur = float(cfg.get('eval_crop_duration', EVAL_CROP_DURATION))
    n_crops = int(cfg.get('test_multi_crops', TEST_MULTI_CROPS))

    fe = AutoFeatureExtractor.from_pretrained(model_name, trust_remote_code=TRUST_REMOTE_CODE)
    mdl = MERTGenreClassifier(model_name, num_classes=len(l2i), dropout=float(cfg.get('head_dropout', HEAD_DROPOUT)))
    mdl.load_state_dict(checkpoint['model_state_dict'])
    mdl.eval()

    fake_rec = TrackRecord(track_id=-1, genre='', split='')

    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='PySoundFile failed. Trying audioread instead.')
            warnings.filterwarnings('ignore', message='librosa.core.audio.__audioread_load')
            y, _ = librosa.load(audio_file, sr=sr, mono=True, duration=max_dur, res_type='kaiser_fast')
    except Exception:
        y = np.zeros(int(sr * crop_dur), dtype=np.float32)

    if y is None or y.shape[0] == 0:
        y = np.zeros(int(sr * crop_dur), dtype=np.float32)
    y = y.astype(np.float32)

    crop_samples = int(round(sr * crop_dur))
    total = y.shape[0]
    starts = [0] if total <= crop_samples else np.linspace(0, total - crop_samples, num=max(1, n_crops), dtype=int).tolist()

    wavs = []
    for s in starts:
        crop = y[s:s+crop_samples]
        if crop.shape[0] < crop_samples:
            crop = np.pad(crop, (0, crop_samples - crop.shape[0]))
        wavs.append(crop.astype(np.float32))

    enc = fe(wavs, sampling_rate=sr, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = mdl(enc.input_values, enc.attention_mask)
        probs = torch.softmax(logits, dim=1).mean(dim=0).cpu().numpy()

    top = np.argsort(probs)[::-1][:top_k]
    return [(i2l[int(i)], float(probs[int(i)])) for i in top]


preds = predict_file('fma_large/002/002003.mp3', OUTPUT_DIR / 'best_model.pt', top_k=5)
for label, p in preds:
    print(f'{label:20s} {p:.4f}')
